# TF-specific GRN importance along human forebrain pseudotime

This focused analysis retains the original manuscript workflow: for selected developmental TFs, it extracts the strongest cell-specific target sets, evaluates their expression coherence against matched control sets and visualizes row-normalized GRN importance along pseudotime.

Run `workflow.ipynb` first or ensure that `results/<result_name>/single_network/` is available. The calculation can be slow; completed TFs are cached in `tutorial_outputs`.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the scDGRN repository.")


REPO_ROOT = find_repository_root()
sys.path.insert(0, str(REPO_ROOT))

DATASET_NAME = "forebrain"
RESULT_NAME = os.environ.get("SCDGRN_RESULT_NAME", "forebrain")
DATASETS_ROOT = Path(os.environ.get("SCDGRN_DATASETS_ROOT", REPO_ROOT / "datasets")).resolve()
RESULTS_ROOT = Path(os.environ.get("SCDGRN_RESULTS_ROOT", REPO_ROOT / "results")).resolve()
DATASET_DIR = DATASETS_ROOT / DATASET_NAME
RESULT_DIR = RESULTS_ROOT / RESULT_NAME
OUTPUT_DIR = RESULT_DIR / "tutorial_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMN = "cell_type"
RANDOM_SEED = 3407
TOP_N_EDGES = 50

print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATASET_DIR}")
print(f"Results:    {RESULT_DIR}")
print(f"Outputs:    {OUTPUT_DIR}")


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm

dataset_name = "forebrain"
specified_tfs = ["SOX2", "HES1", "PAX6", 'EGR1', 'FOS', 'JUN', "EOMES", "SATB2", "SOX11", "SOX4", "SOX5", "BCL11A", "BCL11B"]
max_targets_per_tf = 100
n_controls = 100

project_root = REPO_ROOT
dataset_dir = project_root / "datasets" / dataset_name
network_dir = RESULT_DIR / "single_network"
expression_df = pd.read_csv(dataset_dir / "ExpressionData.csv", index_col=0)
expression_matrix = expression_df.to_numpy(dtype=np.float32)
gene_names = expression_df.columns.to_numpy().astype(str)
all_genes = gene_names.tolist()

cell_labels = pd.read_csv(dataset_dir / "cell_data.csv").values.flatten().astype(str)
pseudotime = pd.read_csv(dataset_dir / "pseudotime.csv")["pseudotime"].to_numpy(dtype=float)

n_cells = expression_matrix.shape[0]
if len(cell_labels) != n_cells or len(pseudotime) != n_cells:
    raise ValueError("Cell counts are inconsistent across expression, labels, or pseudotime.")

tf_network_list = np.genfromtxt(dataset_dir / "TF.csv", delimiter="\n", dtype="str").astype(str)
gene_to_index = {gene: idx for idx, gene in enumerate(gene_names)}
tf_network_to_index = {tf: idx for idx, tf in enumerate(tf_network_list)}

selected_tfs = [tf for tf in specified_tfs if tf in tf_network_to_index and tf in gene_to_index]
missing_tfs = [tf for tf in specified_tfs if tf not in selected_tfs]
if missing_tfs:
    print("Skipped TFs missing from TF network or expression:", missing_tfs)
if not selected_tfs:
    raise ValueError("No specified TF was found in TF network and expression data.")

selected_tf_network_indices = np.array([tf_network_to_index[tf] for tf in selected_tfs], dtype=int)

sys.path.append(str((project_root / "utils").resolve()))
from Grn_Importance import Grn_Importance

calculator = Grn_Importance(expression_matrix, all_genes)


def load_cell_network(cell_idx: int) -> sp.csr_matrix:
    data = np.load(network_dir / f"cell{cell_idx}.npz")
    return sp.coo_matrix(
        (data["data"], (data["row"], data["col"])),
        shape=data["shape"],
    ).tocsr()


def top_targets_for_tf(cell_network: sp.csr_matrix, tf_gene_idx: int, max_targets: int = 100) -> list[str]:
    row = cell_network.getrow(tf_gene_idx)
    if row.nnz == 0:
        return []

    keep_mask = row.indices != tf_gene_idx
    target_indices = row.indices[keep_mask]
    target_weights = row.data[keep_mask]
    if target_indices.size == 0:
        return []

    order = np.argsort(np.abs(target_weights))[::-1][:max_targets]
    return gene_names[target_indices[order]].tolist()


print(f"Expression matrix shape: {expression_matrix.shape}")
print(f"Using {len(selected_tfs)} TFs: {selected_tfs}")
print(f"TF network matrix shape: ({len(tf_network_list)}, {len(gene_names)})")


In [ ]:
legacy_results_path = OUTPUT_DIR / f"{dataset_name}_specified_tf_activity_grn_importance.csv"
results_path = OUTPUT_DIR / f"{dataset_name}_specified_tf_grn_importance.csv"
results_columns = ["cell_index", "cell_label", "pseudotime", "tf", "n_targets", "grn_importance", "p_value"]

cached_frames = []
cached_tfs = set()
for cache_path in [results_path, legacy_results_path]:
    if not cache_path.exists():
        continue

    cached_df = pd.read_csv(cache_path)
    if "tf" not in cached_df.columns or "cell_index" not in cached_df.columns:
        continue

    for tf in selected_tfs:
        if tf in cached_tfs:
            continue

        tf_cache = cached_df.loc[cached_df["tf"] == tf].copy()
        if len(tf_cache) != n_cells or tf_cache["cell_index"].nunique() != n_cells:
            continue

        tf_cache = tf_cache.sort_values("cell_index")
        if tf_cache["cell_index"].to_numpy().tolist() != list(range(n_cells)):
            continue

        tf_cache["cell_label"] = cell_labels
        tf_cache["pseudotime"] = pseudotime
        for col in results_columns:
            if col not in tf_cache.columns:
                tf_cache[col] = np.nan

        cached_frames.append(tf_cache[results_columns])
        cached_tfs.add(tf)
        print(f"Loaded cached GRN importance for {tf} from {cache_path.name}")

tfs_to_compute = [tf for tf in selected_tfs if tf not in cached_tfs]
print(f"Reusing {len(cached_tfs)} cached TFs, computing {len(tfs_to_compute)} TFs: {tfs_to_compute}")

computed_frames = []
if tfs_to_compute:
    tf_index_map = dict(zip(selected_tfs, selected_tf_network_indices))
    cell_target_genes_by_tf = {tf: [[] for _ in range(n_cells)] for tf in tfs_to_compute}

    for cell_idx in tqdm(range(n_cells), desc="Collecting top targets"):
        cell_network = load_cell_network(cell_idx)
        for tf in tfs_to_compute:
            cell_target_genes_by_tf[tf][cell_idx] = top_targets_for_tf(
                cell_network,
                tf_index_map[tf],
                max_targets=max_targets_per_tf,
            )

    for tf in tfs_to_compute:
        print(f"Computing GRN importance for {tf} with Grn_Importance.py ...")
        cell_target_genes = cell_target_genes_by_tf[tf]
        p_values = calculator.compute_grn_scores_scdrs(cell_target_genes, n_controls=n_controls)
        grn_importance = -np.log10(np.clip(p_values, 1e-300, 1.0))
        target_counts = np.array([len(targets) for targets in cell_target_genes], dtype=int)

        computed_frames.append(
            pd.DataFrame(
                {
                    "cell_index": np.arange(n_cells, dtype=int),
                    "cell_label": cell_labels,
                    "pseudotime": pseudotime,
                    "tf": tf,
                    "n_targets": target_counts,
                    "grn_importance": grn_importance,
                    "p_value": p_values,
                }
            )
        )

results_df = pd.concat(cached_frames + computed_frames, ignore_index=True)
results_df["tf"] = pd.Categorical(results_df["tf"], categories=selected_tfs, ordered=True)
results_df = results_df.sort_values(["tf", "cell_index"]).reset_index(drop=True)
results_df.to_csv(results_path, index=False)
print(f"Saved per-cell results to: {results_path.resolve()}")
results_df.head()


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

selected_tfs = ["SOX2", "HES1", "PAX6", 'EGR1', 'FOS', 'JUN', "EOMES",  "SOX11", "SOX4", "SOX5", "BCL11A", "BCL11B"]

dataset_name = "forebrain"

project_root = REPO_ROOT
dataset_dir = project_root / "datasets" / dataset_name

pseudotime = pd.read_csv(dataset_dir / "pseudotime.csv")["pseudotime"].to_numpy(dtype=float)


results_df = pd.read_csv(OUTPUT_DIR / "forebrain_specified_tf_grn_importance.csv")
results_df["tf"] = pd.Categorical(results_df["tf"], categories=selected_tfs, ordered=True)
results_df = results_df.sort_values(["tf", "cell_index"]).reset_index(drop=True)

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter1d


custom_cmap = LinearSegmentedColormap.from_list(
    "grn_importance", ["#f6e09d", "#f5a65b", "#e64532"]
)

sorted_indices = np.argsort(pseudotime)
sorted_pseudotime = pseudotime[sorted_indices]
smoothing_sigma = max(1, len(sorted_pseudotime) // 150)


def smooth_and_normalize(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if np.isnan(values).all():
        return np.zeros_like(values)

    values = np.nan_to_num(values, nan=0.0)
    smoothed = gaussian_filter1d(values, sigma=smoothing_sigma)
    vmin = smoothed.min()
    vmax = smoothed.max()
    if np.isclose(vmin, vmax):
        return np.zeros_like(smoothed)
    return (smoothed - vmin) / (vmax - vmin)


grn_importance_heatmap = []
for tf in selected_tfs:
    tf_results = results_df.loc[results_df["tf"] == tf].sort_values("cell_index")
    grn_importance_heatmap.append(smooth_and_normalize(tf_results["grn_importance"].to_numpy()[sorted_indices]))

grn_importance_heatmap = np.vstack(grn_importance_heatmap)

fig_height = max(4.5, 0.75 * len(selected_tfs))
fig, ax = plt.subplots(figsize=(10, fig_height), constrained_layout=True)

im = ax.imshow(grn_importance_heatmap, aspect="auto", cmap=custom_cmap, vmin=0, vmax=1)

tick_positions = np.linspace(0, len(sorted_pseudotime) - 1, 6, dtype=int)
tick_labels = [f"{sorted_pseudotime[idx]:.2f}" for idx in tick_positions]

ax.set_title("GRN importance along pseudotime", fontsize=14)
ax.set_yticks(np.arange(len(selected_tfs)))
ax.set_yticklabels(selected_tfs, fontsize=12)
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)
ax.set_xlabel("Pseudotime", fontsize=12)
ax.set_ylabel("TF", fontsize=12)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Row-wise normalized GRN importance", fontsize=11)

heatmap_png = OUTPUT_DIR / f"{dataset_name}_specified_tf_grn_importance_heatmap.png"
heatmap_svg = OUTPUT_DIR / f"{dataset_name}_specified_tf_grn_importance_heatmap.svg"
plt.savefig(heatmap_png, dpi=300, bbox_inches="tight")
plt.savefig(heatmap_svg, bbox_inches="tight")
plt.show()

print(f"Saved heatmaps to: {heatmap_png.resolve()} and {heatmap_svg.resolve()}")


## Interpretation boundary

The score is an expression-coherence measure for targets selected from each inferred GRN. It is not an experimental TF-activity measurement and does not establish causal TF action.
